In [ ]:
import subprocess
import os
import logging
import numpy as np

import sys
sys.path.append("work/SIRF-Contribs/src/notebooks/AIRBI-MRI-recon/")
from stgeorges_utils import change_ismrmrd, to_dicom_folder, LogfileCallback

from sirf.Gadgetron import AcquisitionData, ImageData
from sirf.Gadgetron import AcquisitionModel
from sirf.Gadgetron import AcquisitionDataProcessor
from sirf.Gadgetron import CartesianGRAPPAReconstructor, FullySampledReconstructor
from sirf.Gadgetron import CoilSensitivityData
from sirf.Gadgetron import preprocess_acquisition_data

from cil.optimisation.functions import LeastSquares
from cil.optimisation.functions import ZeroFunction
from cil.optimisation.algorithms import FISTA, CGLS, GD
from cil.plugins.ccpi_regularisation.functions import FGP_TV
from cil.framework import DataContainer as cilDataContainer
from cil.optimisation.operators import LinearOperator
from cil.optimisation.utilities.callbacks import ProgressCallback, TextProgressCallback
import tempfile

from cil.optimisation.functions import L1Sparsity
from cil.optimisation.operators import WaveletOperator
from AbsFunction import FunctionOfAbs


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

command = "siemens_to_ismrmrd"

input_file = "/home/jovyan/work/person2/person2/meas_MID00598_FID133064_pd_tse_fs_cor_uflex_no_spine.dat"
#input_file = "/home/jovyan/work/person2/person2/meas_MID00595_FID133061_pd_tse_fs_cor_uflex_no_spine.dat"

#input_file = "/home/jovyan/work/data/meas_MID00619_FID129157_AI_RECON_SEQD_512_GF4_AX_RL.dat"
#input_file = "/home/jovyan/work/data/meas_MID00614_FID129152_CONVENTIONAL_RECON_SEQD_GF2_AX_RL.dat"

proc_dir = tempfile.mkdtemp(prefix="stgeorges_proc_")

fname = input_file
logger.info(f"Processing file {fname}...")
file_in = fname
file_out = os.path.join(proc_dir, os.path.basename(file_in).replace(".dat", ".h5"))
if os.path.exists(file_out):
    logger.warning(f"Output file {file_out} already exists. Removing it.")
    os.remove(file_out)

out = subprocess.run(
    [command, "-f", file_in, "-o", file_out, "-z", "2", "-M"],
    capture_output=True,
    text=True,
)

logger.info(out.stdout)
logger.error(out.stderr)

# Change ISMRMRD file if needed
file_out_mod = file_out.replace(".h5", "_mod.h5")
change_ismrmrd(file_out, file_out_mod)
logger.info(f"Modified ISMRMRD file saved as {file_out_mod}")

In [ ]:
acq_data = AcquisitionData(file_out_mod)

In [ ]:
print(acq_data.get_header())

In [ ]:
acq_data = preprocess_acquisition_data(acq_data)



In [ ]:
csm = CoilSensitivityData()
csm.smoothness = 100
csm.calculate(acq_data)
csm_array = csm.as_array()
csm_array[np.isnan(csm_array)] = 0
csm_array[np.isinf(csm_array)] = 0
csm.fill(csm_array)

E = AcquisitionModel(acqs=acq_data, imgs=csm)
E.set_coil_sensitivity_maps(csm)
image_data = E.adjoint(acq_data)

In [ ]:
import matplotlib.pyplot as plt

image_array = image_data.as_array()
image_array[np.isnan(image_array)] = 0
image_array = image_array/abs(image_array).max()

fig, ax = plt.subplots(1,2, figsize=(8,4))
centre_slice = image_array.shape[0]//2
ax[0].imshow(abs(image_array[centre_slice,:,:]), vmin=0, vmax=0.7, cmap='gray')
ax[1].imshow(np.angle(image_array[centre_slice,:,:]), vmin=-np.pi, vmax=np.pi, cmap='bwr')
plt.suptitle(input_file, fontsize=6)